# Prop Firm Strategy Simulator

This notebook explains the mathematics behind the historical replay and the two Monte Carlo experiments used by the live model. It is a research simulation, not a forecast of future payouts.

## 1. From TradingView rows to realized trades

The included evaluation and funded CSV files contain both an entry row and an exit row for each completed trade, and TradingView repeats the trade's realized P&L on both rows. The simulator therefore keeps **exit rows only**. If trade $i$ realizes profit or loss $x_i$, account equity evolves as

$$E_i = E_{i-1} + x_i.$$

The two files remain separate because the evaluation strategy is replayed while the account is in evaluation and the funded strategy is replayed only after an evaluation pass.

## 2. Evaluation as a state machine

An account begins in the **evaluation** state with starting balance $B_0$. At every exit, the simulator updates equity, daily P&L, and peak equity. The trailing drawdown after trade $i$ is

$$D_i^{trail}=\max_{j\le i} E_j-E_i.$$

The evaluation fails if the daily loss or trailing drawdown reaches its selected limit. A failed evaluation returns to a fresh evaluation state and charges the selected reset fee.

The profit target is reached when $E_i-B_0\ge T$. Passing also requires a minimum number of trading days and a best-day consistency ratio no larger than the selected limit:

$$C=\frac{\max_d P_d}{\sum_d P_d}\le c,$$

where $P_d$ is realized P&L on day $d$. On a pass, the evaluation account closes and a fresh funded account begins at $B_0$.

## 3. Funded replay and payouts

The funded account uses a static maximum-loss floor and a daily-loss limit. It fails when

$$B_0-E_i\ge L_{funded}$$

or when that day's cumulative loss reaches its selected limit. A funded failure sends the simulation back to evaluation.

Under the standard payout rule, eligibility requires $m$ winning days whose daily profit is at least $w$. The payout is capped at $K$:

$$Payout=\min\left(\sum_d P_d, K\right).$$

The simulator also retains the supplied consistency-based payout alternative. Business-level net result is total payouts less evaluation reset fees:

$$Net=\sum Payouts-(N_{resets}\times Fee).$$

## 4. Historical window

The 1-, 2-, or 3-year choice selects completed trades between the latest timestamp in the included data and the same calendar date the chosen number of years earlier. The simulator then walks both trade streams chronologically. Because rules depend on path, changing the window can change when accounts pass, reset, receive payouts, or fail.

## 5. Monte Carlo without replacement

For each path, the simulator randomly permutes the observed evaluation outcomes and funded outcomes separately, then maps them onto the original timestamps. If $\pi$ is a random permutation, the simulated sequence is

$$x_i^*=x_{\pi(i)}.$$

Every observed outcome appears exactly once, so total raw trade P&L is preserved. What changes is order. This isolates sequence risk: identical trades can produce different account outcomes because daily limits, drawdowns, passes, and payouts are path-dependent.

## 6. Monte Carlo with replacement

Bootstrap resampling draws each outcome independently from the empirical trade distribution:

$$x_i^*\sim \widehat F_n=\frac{1}{n}\sum_{j=1}^{n}\delta_{x_j}.$$

A trade can appear more than once or not at all. This changes both the composition and order of outcomes while retaining the original schedule of timestamps and trading days. It asks a broader question than shuffling: what range of account paths is plausible if the included empirical outcomes are treated as the sampling distribution?

## 7. Reading the output

The live page reports the historical net result and a Monte Carlo distribution. For simulated net results $Y_1,\ldots,Y_M$, it displays the mean, median, empirical 10th and 90th percentiles, and

$$\widehat{Pr}(Y>0)=\frac{1}{M}\sum_{m=1}^{M}\mathbf{1}(Y_m>0).$$

It also averages evaluation passes, payouts, and funded failures across paths. These are conditional results from the supplied logs and chosen rules. They do not include slippage changes, commissions not present in the logs, strategy decay, platform execution risk, or changes to a prop firm's actual terms.

In [ ]:
# Core implementation used by the hosted page
from math_code.prop_sim.rules import DEFAULT_RULES
from math_code.prop_sim.simulator import run_simulation
from math_code.prop_sim.monte_carlo import run_monte_carlo_summaries

DEFAULT_RULES